# 4. Backtesting and forecast visualisation

**Stage 1, Step 5 — sections 7, 33.**

Two questions:

1. **Is the accuracy stable over time**, or was the test fold a lucky window?
2. **Do the prediction intervals cover what they claim?** A 90% interval that covers 71% is a finding, not a detail — and the only way to know is to measure it on data the calibration never saw.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from app.services.container import Container
from ml.forecasting.backtest import backtest_by_horizon, stability_by_bucket
from ml.forecasting.baselines import attach_seasonal_reference
from ml.forecasting.config import load_forecast_config
from ml.forecasting.conformal import measure_horizon_coverage
from ml.forecasting.dataset import (
    HORIZON_STEP,
    TARGET,
    TARGET_DATE,
    HorizonDataset,
    build_history,
    build_horizon_dataset,
)
from ml.forecasting.evaluate import PREDICTED, hierarchy_table
from ml.forecasting.sampling import sample_series
from ml.forecasting.split import build_origin_split, slice_fold
from ml.forecasting.train import build_estimator, train_forecaster

repo = Container().data_repository
config = load_forecast_config().smoke()

sample = sample_series(repo, n_series=config.sampling.n_series, seed=config.sampling.seed)
history = build_history(repo, config, sample)
view = repo.as_of(pd.to_datetime(history["date"]).dt.date.max())
dataset = build_horizon_dataset(history, view, config, sample)
dataset = HorizonDataset(
    frame=attach_seasonal_reference(dataset.frame, history),
    feature_names=[*dataset.feature_names, "seasonal_reference"],
    excluded=dataset.excluded,
)
split = build_origin_split(dataset.frame, config)
model = train_forecaster(dataset, build_estimator("lightgbm", seed=42), config, split)

test = slice_fold(dataset.frame, split.test_start, split.test_end)
scored = test.assign(**{PREDICTED: model.predict(test)})
print(f"scored {len(scored):,} test rows")

## 1. Walk-forward stability

Expanding-window over **origins**, with the same embargo the main split uses. Dropping the embargo here while keeping it there would make backtest numbers systematically better than the test number — the sort of inconsistency that gets explained away as "the backtest is optimistic" rather than fixed.

In [ ]:
folds = backtest_by_horizon(
    dataset, lambda: build_estimator("lightgbm", seed=42), config
)
stability = stability_by_bucket(folds)

display(stability.style.format({"mean": "{:.1%}", "std": "{:.2%}"}))

unstable = stability[~stability["stable"]]
if not unstable.empty:
    print()
    print("Buckets whose accuracy varies materially between folds:")
    print(unstable[["bucket", "mean", "std"]].to_string(index=False))
    print("A single headline number understates the uncertainty there.")

## 2. Actual vs forecast (§33)

In [ ]:
daily = (
    scored.groupby(TARGET_DATE, observed=True)
    .agg(actual=(TARGET, "sum"), forecast=(PREDICTED, "sum"))
    .reset_index()
    .sort_values(TARGET_DATE)
)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].plot(daily[TARGET_DATE], daily["actual"], label="actual", linewidth=1.2)
axes[0].plot(daily[TARGET_DATE], daily["forecast"], label="forecast", linewidth=1.2, linestyle="--")
axes[0].set_ylabel("units")
axes[0].set_title("Actual vs forecast, aggregated across the test panel")
axes[0].legend()

error = daily["forecast"] - daily["actual"]
axes[1].bar(daily[TARGET_DATE], error, width=1.0,
            color=np.where(error >= 0, "tab:orange", "tab:blue"))
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("forecast - actual")
axes[1].set_title("Forecast error (orange = over-forecast)")

plt.tight_layout()
plt.show()

print(f"aggregate bias: {error.sum() / daily['actual'].sum():+.2%}")

Bias matters more than dispersion here. Random error averages out over a planning period; a consistent skew does not, and it compounds into every inventory decision built on the forecast.

## 3. Prediction intervals — measured, not claimed

In [ ]:
coverage = measure_horizon_coverage(
    scored, model.calibration, config, actual_column=TARGET, predicted_column=PREDICTED
)

rows = [
    {
        "bucket": bucket,
        "nominal": report.nominal,
        "measured": report.empirical,
        "gap": report.gap,
        "width_vs_actual": report.relative_width,
        "n": report.n,
        "calibrated": report.is_calibrated,
    }
    for bucket, report in coverage.items()
]
table = pd.DataFrame(rows)
display(table.style.format({
    "nominal": "{:.0%}", "measured": "{:.1%}", "gap": "{:+.1%}",
    "width_vs_actual": "{:.0%}",
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(table["bucket"], table["measured"])
axes[0].axhline(table["nominal"].iloc[0], color="red", linestyle="--", label="nominal")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Measured coverage by horizon bucket")
axes[0].legend()

axes[1].bar(table["bucket"], table["width_vs_actual"], color="tab:orange")
axes[1].set_title("Interval width as a share of mean actual")

plt.tight_layout()
plt.show()

Coverage is reported **per bucket and never blended**. A blended figure can sit at a healthy 90% while short horizons cover 98% and long ones cover 71% — and it is the long ones a planner is least able to sanity-check by eye.

Note also that the width grows with horizon only because each bucket has its own calibration. A single global quantile would produce a width proportional to the prediction alone, which barely widens with *h*.

## 4. Forecast during promotion and during stockout (§33)

In [ ]:
promo_column = "h_promotion_flag"
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if promo_column in scored.columns:
    promoted = scored[scored[promo_column].astype(bool)]
    clean = scored[~scored[promo_column].astype(bool)]
    axes[0].bar(
        ["no promotion", "promotion planned"],
        [
            clean[PREDICTED].sum() / clean[TARGET].sum(),
            promoted[PREDICTED].sum() / promoted[TARGET].sum(),
        ],
    )
    axes[0].axhline(1.0, color="black", linestyle="--")
    axes[0].set_ylabel("forecast / actual")
    axes[0].set_title("Does the model use the promotion plan?")
    print(f"promotional rows in test: {len(promoted):,}")

by_bucket = (
    scored.assign(bucket=scored[HORIZON_STEP].map(config.intervals.bucket_for))
    .groupby("bucket", observed=True)
    .apply(lambda b: (b[PREDICTED].sum() - b[TARGET].sum()) / b[TARGET].sum())
)
axes[1].bar(by_bucket.index, by_bucket.to_numpy())
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("bias")
axes[1].set_title("Forecast bias by horizon bucket")

plt.tight_layout()
plt.show()

Note what is **not** shown: a stockout comparison. Rows whose target fell on a stockout were excluded from training *and* are excluded from scoring, because the target there records availability rather than demand. Plotting forecast-versus-actual on those rows would compare a demand forecast against a censored number and make the model look badly biased for the wrong reason.

That exclusion has a real cost, and it is stated rather than hidden: stockouts are endogenous — latent demand during one runs about 1.57× normal — so dropping them removes part of the high-demand tail.

## 5. Accuracy by aggregation level

In [ ]:
hierarchy = hierarchy_table(scored)
display(hierarchy.style.format({"wmape": "{:.1%}", "bias_pct": "{:+.1%}", "mae": "{:.1f}"}))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(hierarchy["level"], hierarchy["wmape"])
ax.set_ylabel("WMAPE")
ax.set_title("Error falls as you aggregate - the price of bottom-up coherence")
plt.tight_layout()
plt.show()

Bottom-up aggregation is **exactly coherent** — the regional number is always the sum of its store numbers — so no reconciliation is applied.

What this shows is the price of that choice: independent errors average out as you aggregate. It answers the question a planner actually asks — *should I trust the regional figure more than the SKU figure?* — with a magnitude rather than a shrug.

---

## Findings

- Accuracy is checked fold by fold and bucket by bucket; instability is reported rather than averaged away.
- Interval coverage is **measured** per bucket. Whatever it turns out to be is what gets reported.
- Error falls monotonically as forecasts are aggregated, which quantifies how much more the regional number can be trusted.

### Next

`05_error_analysis.ipynb` — which segments are worst, and is the error biased anywhere it matters?